# 07 · Multivariate Anomaly Detection

## What this notebook covers
Most anomaly detection methods look at each feature independently. Multivariate methods capture *correlations* between features — a point can be perfectly normal on each dimension individually but anomalous in the joint space.

## The key intuition
Suppose CPU=90% and memory=90% individually appear sometimes. But CPU=90% *with* memory=5% never appears together under normal conditions — the correlation is violated. Univariate methods miss this; multivariate methods catch it.

## Methods implemented

**Mahalanobis distance** is the multivariate generalisation of z-score. It measures distance in terms of the data's covariance structure: a point is anomalous if it is far from the mean *relative to the spread of the data in that direction*. It assumes multivariate normality — excellent when that assumption holds, degrades otherwise.

**COPOD** (Copula-Based Outlier Detection): models the joint distribution using copulas — separating marginal distributions from the dependence structure. It is parameter-free, fast, and handles non-normal data well. Implemented in the PyOD library.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import EmpiricalCovariance, MinCovDet
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
np.random.seed(42)

# Multivariate normal with correlation structure
n_normal = 500
cov = np.array([[1.0, 0.8, 0.3],
                [0.8, 1.0, 0.1],
                [0.3, 0.1, 1.0]])
X_normal = np.random.multivariate_normal([0, 0, 0], cov, n_normal)

# Anomalies: violate the correlation structure
X_anom_corr = np.column_stack([
    np.random.normal(0, 1, 20),
    np.random.normal(0, 1, 20) * -1,  # inverted correlation
    np.random.normal(0, 1, 20),
])
# Anomalies: extreme marginal values
X_anom_extreme = np.random.multivariate_normal([4, 4, 4], np.eye(3), 10)

X_all  = np.vstack([X_normal, X_anom_corr, X_anom_extreme])
y_all  = np.array([0]*n_normal + [1]*20 + [1]*10)
print(f"Dataset: {len(X_all)} points  |  {y_all.sum()} anomalies")

In [ ]:
# ── Mahalanobis distance ───────────────────────────────────────────────────────
# Robust covariance estimation (MinCovDet is more robust to outliers than EmpiricalCovariance)
rcd = MinCovDet(random_state=0)
rcd.fit(X_normal)
maha = rcd.mahalanobis(X_all)  # squared Mahalanobis distance

# Under normality, Mahalanobis^2 ~ Chi^2(p)
p_values = 1 - stats.chi2.cdf(maha, df=X_all.shape[1])
auc_maha = roc_auc_score(y_all, maha)
print(f"Mahalanobis ROC-AUC: {auc_maha:.4f}")

In [ ]:
# ── COPOD via PyOD ─────────────────────────────────────────────────────────────
try:
    from pyod.models.copod import COPOD
    copod = COPOD(contamination=0.06)
    copod.fit(X_all)
    scores_copod = copod.decision_scores_
    auc_copod = roc_auc_score(y_all, scores_copod)
    print(f"COPOD ROC-AUC: {auc_copod:.4f}")
    has_pyod = True
except ImportError:
    print("PyOD not installed — install with: pip install pyod")
    print("Using Mahalanobis only for visualisation.")
    scores_copod = maha
    has_pyod = False

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 3D projected to 2D (first two features)
sc = axes[0].scatter(X_all[:, 0], X_all[:, 1], c=np.sqrt(maha), cmap="RdYlGn_r", s=15, alpha=0.7)
axes[0].scatter(X_all[y_all==1, 0], X_all[y_all==1, 1], c="red", s=60, marker="x", zorder=5, label="True anomaly")
plt.colorbar(sc, ax=axes[0], label="√Mahalanobis")
axes[0].set_title(f"Mahalanobis distance (AUC={auc_maha:.3f})")
axes[0].legend(fontsize=8)

# Score distributions
for scores, label, color in [(np.sqrt(maha), "Mahalanobis", "steelblue")]:
    axes[1].hist(scores[y_all==0], bins=30, alpha=0.6, color=color, label=f"{label} normal", density=True)
    axes[1].hist(scores[y_all==1], bins=15, alpha=0.6, color="tomato", label=f"Anomaly", density=True)
threshold_m = np.percentile(np.sqrt(maha[y_all==0]), 95)
axes[1].axvline(threshold_m, color="black", linestyle="--", label=f"95th pct = {threshold_m:.2f}")
axes[1].set_title("Score distribution")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{base}/07_multivariate.png", dpi=120)
plt.show()